In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Ce fichier nétoie et fusionne les donnés financier en un seul dataset

In [4]:

df0=pd.DataFrame()
for i in range(18,23):
    df=pd.read_excel(f"file20{i}.xlsx",header=[0,2])
    col=list(df.columns)
    col[0]=(col[0][0],'N°')
    df.columns=col
    df0=pd.concat([df0,df])

def is_good(val):
    return isinstance(val, (int, float))

df0=df0[df0[('CHARGES EXCEPTIONNELLES', 'Total des produits (I + III + V)')].apply(is_good)]
df0=df0[df0[('CHARGES EXCEPTIONNELLES', 'Total des charges (II + IV +VI + VII + VIII + IX)')].apply(is_good)]

new_col=list(df0.columns)
for j, i in enumerate(new_col):
    if 'Unnamed' in i[0]:
        new_col[j]=i[1]
    else:
        new_col[j]=(i[0].lower(),i[1].lower())
    if '(brut)' in i[1] or '(amortissements et dépréciations)' in i[1]:
        df0.drop(columns=[i],inplace=True)
        new_col[j]='0'
    elif '(net)' in i[1]:
        new_col[j]=(new_col[j][0],new_col[j][1].replace(' (net)',''))

new_col=[k for k in new_col if k!='0']
new_col[0]='num parti'
new_col[1]='nom parti'
new_col[3]='année'
df0.columns=new_col
df0.drop(index=df0[~df0['Unité monétaire'].isin(['Euro', 'EUR', 'euro'])].index, inplace=True)

df0.drop(columns=['Unité monétaire'],inplace=True)
df0.fillna(0)
df0['total flux']=df0[('charges exceptionnelles', 'total des produits (i + iii + v)')] + df0[('charges exceptionnelles', 'total des charges (ii + iv +vi + vii + viii + ix)')]


C:\Users\math\AppData\Local\Temp\ipykernel_12772\3220629833.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df0.fillna(0)


In [5]:
df0

,num parti,nom parti,année,"(actif immobilisé, écarts d'acquisition)","(actif immobilisé, immobilisations incorporelles)","(actif immobilisé, terrains et constructions)","(actif immobilisé, autres immobilisations corporelles)","(actif immobilisé, participations et créances rattachées)","(actif immobilisé, autres titres immobilisés)","(actif immobilisé, prêts à des partis ou groupements politiques)",...,"(charges exceptionnelles, dotations aux amortissements, aux dépréciations et aux provisions excep.)","(charges exceptionnelles, total vi)","(charges exceptionnelles, 4. résultat exceptionnel (v - vi))","(charges exceptionnelles, impôts sur les bénéfices (vii))","(charges exceptionnelles, dotations aux amortissements des écarts d'acquisition (viii))","(charges exceptionnelles, intérêts des tiers (ix))","(charges exceptionnelles, total des produits (i + iii + v))","(charges exceptionnelles, total des charges (ii + iv +vi + vii + viii + ix))","(charges exceptionnelles, excédent ou déficit d'ensemble)",total flux
0,5,L'ALLIANCE RÉGIONALE,2018,0,0,0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,77652,61186,16466,138838
1,13,CENTRE NATIONAL DES INDÉPENDANTS ET PAYSANS,2018,0,0,25529,3068.0,0,0,0,...,0.0,0.0,80.0,0.0,0.0,0.0,85495,97354,-11859,182849
2,40,RASSEMBLEMENT NATIONAL,2018,0,3849,59760,336347.0,14000,39160,726201,...,17985.0,1424513.0,816923.0,0.0,0.0,0.0,11690024,14099743,-2409719,25789767
3,42,GÉNÉRATION ÉCOLOGIE,2018,0,0,0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,19723,16062,3661,35785
4,46,GROUPEMENT FRANCE-RÉUNION,2018,0,0,0,0.0,0,15,0,...,0.0,495.0,-495.0,0.0,0.0,0.0,13565,11289,2276,24854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546,1455,TERRE D'AVENIR,2022,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,12900,3674.9,9225.1,16574.9
547,1457,100% VESOUL,2022,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
548,1464,MAISON COMMUNE,2022,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
549,1473,LOIRE-ATLANTIQUE À GAUCHE,2022,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0


In [6]:
correspondance = {
    # Correspondances directes
    'num parti':'num parti',
    'nom parti':'nom parti',
    'année':'année',
    ('i. - actif immobilisé', 'immobilisations incorporelles'): ('actif immobilisé', 'immobilisations incorporelles'),
    ('i. - actif immobilisé', "ecart d'acquisition"): ('actif immobilisé', "écarts d'acquisition"),
    ('i. - actif immobilisé', '- terrains et constructions'): ('actif immobilisé', 'terrains et constructions'),
    ('i. - actif immobilisé', '- autres immobilisations corporelles'): ('actif immobilisé', 'autres immobilisations corporelles'),
    ('i. - actif immobilisé', 'participations'): ('actif immobilisé', 'participations et créances rattachées'),
    ('i. - actif immobilisé', '- prêts'): ('actif immobilisé', 'prêts à des partis ou groupements politiques'),
    ('i. - actif immobilisé', '- autres immobilisations financières'): ('actif immobilisé', 'autres immobilisations financières'),
    ('ii. - actif circulant', 'stock et en-cours'): ('actif circulant', 'stocks et en-cours'),
    ('ii. - actif circulant', 'créances :'): ('actif circulant', 'créances clients et comptes rattachés'),
    ('ii. - actif circulant', '- autres créances'): ('actif circulant', 'autres créances'),
    ('ii. - actif circulant', 'valeurs mobilières de placement'): ('actif circulant', 'valeurs mobilières de placement'),
    ('ii. - actif circulant', 'disponibilités'): ('actif circulant', 'disponibilités'),
    ('iii. - comptes de régularisation', "charges constatées d'avance"): ('actif circulant', "charges constatées d'avance"),
    ('iii. - comptes de régularisation', "total de l'actif"): ('actif circulant', 'total général (i + ii + iii)'),
    ("i. - fonds propres de l'ensemble", 'réserves :'): ('fonds propres', 'réserves'),
    ("i. - fonds propres de l'ensemble", "excédent ou perte de l'exercice"): ('fonds propres', "excédent ou déficit de l'exercice"),
    ('ii. - provisions pour risques et charges', 'provisions pour risques'): ('provisions', 'provisions pour risques'),
    ('ii. - provisions pour risques et charges', 'provisions pour charges'): ('provisions', 'provisions pour charges'),
    ('iii. - dettes', 'emprunts et dettes auprès des établissements de crédit'): ('dettes', "emprunts et dettes auprès d'établissement de crédit"),
    ('iii. - dettes', 'dettes fournisseurs et comptes rattachés'): ('dettes', 'dettes fournisseurs et comptes rattachés'),
    ('iii. - dettes', 'dettes fiscales et sociales'): ('dettes', 'dettes fiscales et sociales'),
    ('iii. - dettes', 'autres dettes'): ('dettes', 'autres dettes'),
    ('iii. - comptes de régularisation', "produits constatés d'avance"): ('dettes', "produits constatés d'avance"),
    ('iii. - comptes de régularisation', 'total du passif'): ('dettes', 'total général (i + ii + iii + iv)'),
    ('cr: charges', 'propagande et communication'): ("charges d'exploitation", 'communication (presse, publications, télévisions, publicité, sites internet, réseaux sociaux)'),
    ('cr: charges', 'aides financières aux candidats :'): ("charges d'exploitation", 'contributions versées aux candidats'),
    ('cr: charges', 'autres aides financières'): ("charges d'exploitation", 'contributions à des partis ou groupements politiques'),
    ('cr: charges', 'achats consommés'): ("charges d'exploitation", 'achats de marchandises et variation de stocks'),
    ('cr: charges', 'autres charges externes'): ("charges d'exploitation", 'autres achats et autres charges externes'),
    ('cr: charges', 'impôts et taxes'): ("charges d'exploitation", 'impôts et taxes'),
    ('cr: charges', '- salaires'): ("charges d'exploitation", 'salaires et traitements'),
    ('cr: charges', '- charges sociales'): ("charges d'exploitation", 'charges sociales'),
    ('cr: charges', 'charges financières'): ('charges financières', 'total iv'),
    ('cr: charges', 'charges exceptionnelles'): ('charges exceptionnelles', 'total vi'),
    ('cr: charges', '- dotation aux amortissements des charges à répartir'): ("charges d'exploitation", 'dotations aux amortissements et dépréciations sur immobilisations'),
    ('cr: charges', '- dotation aux provisions pour campagnes électorales'): ("charges d'exploitation", 'dotations aux provisions'),
    ('cr: charges', 'total des charges'): ('charges exceptionnelles', 'total des charges (ii + iv +vi + vii + viii + ix)'),
    ('cr: produits', 'cotisations des adhérents'): ("produits d'exploitation", 'cotisations des adhérents'),
    ('cr: produits', 'contributions des élus'): ("produits d'exploitation", 'cotisations des élus'),
    ('cr: produits', 'financement public'): ("produits d'exploitation", 'aide publique 1ère fraction'),
    ('cr: produits', '-dont deuxième fraction'): ("produits d'exploitation", 'aide publique 2nde fraction'),
    ('cr: produits', 'dons de personnes physiques'): ("produits d'exploitation", 'dons de personne physique'),
    ('cr: produits', 'dévolution de l’excédent des comptes de campagne'): ("produits d'exploitation", "dévolutions de l'excédent des comptes de campagne"),
    ('cr: produits', 'contributions reçues d’autres formations politiques'): ("produits d'exploitation", 'dévolutions de partis ou groupements politiques'),
    ('cr: produits', 'produits de manifestations et colloques'): ("produits d'exploitation", 'prestations de services (manifestations et colloques)'),
    ('cr: produits', 'produits d’exploitation'): ("produits d'exploitation", 'ventes de marchandises, productions vendue (biens et services), production stockée et production immobilisée'),
    ('cr: produits', 'produits financiers'): ('produits financiers', 'total iii'),
    ('cr: produits', 'produits exceptionnels'): ('produits exceptionnels', 'total v'),
    ('cr: produits', 'reprise sur provisions et amortissements'): ("produits d'exploitation", 'reprise sur amortissements, dépréciations, provisions et transferts de charges'),
    ('cr: produits', 'total des produits'): ('charges exceptionnelles', 'total des produits (i + iii + v)'),

    # Correspondances dérivées
    'résultat flux monétaire': ('charges exceptionnelles', "excédent ou déficit d'ensemble"),
    'total flux monétaire': 'total flux'
}


In [7]:
df02=df0[list(correspondance.values())]

In [8]:
datas=[]
df=pd.DataFrame()
for i in range(8,18):
    if i<15:
        df0=pd.read_excel(f"file{2000+i}.xlsx",header=[0,1])
    else:
        df0=pd.read_excel(f"file{2000+i}.ods",header=[0,1])
    column=list(df0.columns)
    column[0]='Num parti'
    column[1]=('Nom parti')
    column[2]=('Unité monétaire', 'euro')
    column[next(i for i, col in enumerate(column) if col[0] == 'CR: PRODUITS' and col[1].startswith('Financement public'))]=('CR: PRODUITS', 'Financement public')
    column=[(i[0].strip().lower(),i[1].strip().lower()) if type(i)==tuple else i.strip().lower() for i in column]

    df0.fillna(0,inplace=True)
    df0.columns=column

    if ('CR: CHARGES', 'Charges de personnel') in column:
        df0[('CR: CHARGES', 'Charges exceptionnelles')]+=df0[('CR: CHARGES', 'Charges de personnel')]
        df0.drop(columns=[('CR: CHARGES', 'Charges de personnel')],inplace=True)
    df0['année']=2000+i
   
    df=pd.concat([df,df0],ignore_index=True)

del(df0)


df.drop(index=list(df[df[('unité monétaire', 'euro')]!='euro'].index),inplace=True)
df.drop(columns=[('unité monétaire', 'euro')],inplace=True)

df[('résultat flux monétaire')]=df[('cr: produits', 'total des produits')]-df[('cr: charges', 'total des charges')]
df[('total flux monétaire')]=abs(df[('cr: produits', 'total des produits')])+abs(df[('cr: charges', 'total des charges')])
#df[('CR: CHARGES', 'Dotations aux amortissements et provisions (à saisir si différent de la somme des deux suivants)')]=df[('CR: CHARGES', '- dotation aux amortissements des charges à répartir')]+df0[('CR: CHARGES', '- dotation aux provisions pour campagnes électorales')]
df.fillna(0,inplace=True)
df.drop(columns=[('cr: charges', 'dotations aux amortissements et provisions (à saisir si différent de la somme des deux suivants)'),('cr: charges', 'total'),('cr: produits', 'total'),('cr: produits', "résultat d'ensemble (perte)"),('cr: charges', "résultat d'ensemble (excédent)")],inplace=True)
df[('ii. - provisions pour risques et charges', 'provisions pour charges')]=df[('ii. - provisions pour risques et charges', 'provisions pour campagnes électorales')]+df[('ii. - provisions pour risques et charges', 'provisions pour autres charges')]

In [9]:
df=df[list(correspondance.keys())]

In [10]:
col=list(df.columns)
for i in range(len(col)):
    col[i]=correspondance[col[i]]
df.columns=col

In [11]:
df=pd.concat([df,df02])

In [12]:
df

,num parti,nom parti,année,"(actif immobilisé, immobilisations incorporelles)","(actif immobilisé, écarts d'acquisition)","(actif immobilisé, terrains et constructions)","(actif immobilisé, autres immobilisations corporelles)","(actif immobilisé, participations et créances rattachées)","(actif immobilisé, prêts à des partis ou groupements politiques)","(actif immobilisé, autres immobilisations financières)",...,"(produits d'exploitation, dévolutions de l'excédent des comptes de campagne)","(produits d'exploitation, dévolutions de partis ou groupements politiques)","(produits d'exploitation, prestations de services (manifestations et colloques))","(produits d'exploitation, ventes de marchandises, productions vendue (biens et services), production stockée et production immobilisée)","(produits financiers, total iii)","(produits exceptionnels, total v)","(produits d'exploitation, reprise sur amortissements, dépréciations, provisions et transferts de charges)","(charges exceptionnelles, total des produits (i + iii + v))","(charges exceptionnelles, excédent ou déficit d'ensemble)",total flux
0,1,DEMOCRATIE ET REPUBLIQUE (anciennement Metz po...,2008,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,1905.0,0.0,0.0,186008.0,-1007.0,373023.0
1,5,L'alliance régionale,2008,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,70921.0,-781.0,142623.0
2,13,Centre national des Indépendants et Paysans,2008,0,0.0,124651.0,4982.0,0.0,0.0,840.0,...,0.0,0.0,88800,1489.0,27433.0,0.0,0.0,154684.0,-52223.0,361591.0
3,17,COMBAT POUR LES VALEURS,2008,0,0.0,0.0,0.0,0.0,0.0,30.0,...,0.0,0.0,0,31386.0,0.0,0.0,0.0,31686.0,-14573.0,77945.0
4,40,FRONT NATIONAL,2008,24537,0.0,4198171.0,741272.0,14000.0,717658.0,93728.0,...,0.0,15000.0,196224,206599.0,45986.0,86966.0,163809.0,3854207.0,-2445426.0,10153840.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546,1455,TERRE D'AVENIR,2022,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12900,9225.1,16574.9
547,1457,100% VESOUL,2022,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0
548,1464,MAISON COMMUNE,2022,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0
549,1473,LOIRE-ATLANTIQUE À GAUCHE,2022,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0


In [9]:
df[df['num parti']==40]['nom parti'].unique()

array(['FRONT NATIONAL', 'RASSEMBLEMENT NATIONAL'], dtype=object)

In [10]:
del column,correspondance,datas,df02,new_col

In [11]:
col=list(df.columns)
for i in range(len(col)):
    if type(col[i])==tuple:
        col[i]=col[i][1]
df.columns=col

In [12]:
def convert(val):
    try:
        return int(val)
    except:
        print(f"'{val}' converted to 0")
        return 0

In [13]:
df['immobilisations incorporelles'].apply(convert)

' ' converted to 0
'                          -  ' converted to 0
'nan' converted to 0


0          0
1          0
2          0
3          0
4      24537
       ...  
546        0
547        0
548        0
549        0
550        0
Name: immobilisations incorporelles, Length: 5443, dtype: int64

In [14]:

for i in list(df.columns[3:]):
    #print(i)
    df[i]=df[i].apply(convert)


' ' converted to 0
'                          -  ' converted to 0
'nan' converted to 0
'                        -  ' converted to 0
'nan' converted to 0
'                         -  ' converted to 0
'nan' converted to 0
'                         -  ' converted to 0
'                         -  ' converted to 0
'nan' converted to 0
'                        -  ' converted to 0
'                  -  ' converted to 0
'                  -  ' converted to 0
'                         -  ' converted to 0
'nan' converted to 0
'nan' converted to 0
',' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'                                                            ' converted to 0
' ' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
' ' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' converted to 0
'nan' 

In [15]:
df.dtypes

num parti                                                                                                        int64
nom parti                                                                                                       object
année                                                                                                            int64
immobilisations incorporelles                                                                                    int64
écarts d'acquisition                                                                                             int64
terrains et constructions                                                                                        int64
autres immobilisations corporelles                                                                               int64
participations et créances rattachées                                                                            int64
prêts à des partis ou groupements politiques    

In [ ]:
#df.to_csv('finances08_22.csv',index=False)

In [ ]:
#dx=pd.read_csv('finances08_22.csv')

In [ ]:
#dx

,num parti,nom parti,année,immobilisations incorporelles,écarts d'acquisition,terrains et constructions,autres immobilisations corporelles,participations et créances rattachées,prêts à des partis ou groupements politiques,autres immobilisations financières,...,dévolutions de l'excédent des comptes de campagne,dévolutions de partis ou groupements politiques,prestations de services (manifestations et colloques),"ventes de marchandises, productions vendue (biens et services), production stockée et production immobilisée",total iii,total v,"reprise sur amortissements, dépréciations, provisions et transferts de charges",total des produits (i + iii + v),excédent ou déficit d'ensemble,total flux
0,1,DEMOCRATIE ET REPUBLIQUE (anciennement Metz po...,2008,0,0,0,0,0,0,0,...,0,0,0,0,1905,0,0,186008,-1007,373023
1,5,L'alliance régionale,2008,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,70921,-781,142623
2,13,Centre national des Indépendants et Paysans,2008,0,0,124651,4982,0,0,840,...,0,0,88800,1489,27433,0,0,154684,-52223,361591
3,17,COMBAT POUR LES VALEURS,2008,0,0,0,0,0,0,30,...,0,0,0,31386,0,0,0,31686,-14573,77945
4,40,FRONT NATIONAL,2008,24537,0,4198171,741272,14000,717658,93728,...,0,15000,196224,206599,45986,86966,163809,3854207,-2445426,10153840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5438,1455,TERRE D'AVENIR,2022,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12900,9225,16574
5439,1457,100% VESOUL,2022,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5440,1464,MAISON COMMUNE,2022,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5441,1473,LOIRE-ATLANTIQUE À GAUCHE,2022,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
